<a href="https://colab.research.google.com/github/Tejashree-Khot/medsiglip-448-finetune/blob/develop/medsiglip_448_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from collections import OrderedDict
from pathlib import Path
from typing import Callable, cast

import kagglehub
import numpy as np
import pandas as pd
import torch
import wandb
from google.colab import userdata
from huggingface_hub import login
from matplotlib import pyplot as plt
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tensorflow.image import resize as tf_resize
from termcolor import colored
from torch import Tensor, nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from torchvision.models import EfficientNet_B0_Weights
from tqdm import tqdm
from transformers import AutoModel, AutoProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cpu


In [3]:
# Retrieve secret
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
path = kagglehub.dataset_download("mariaherrerot/idrid-dataset")
print("Path to dataset files:", path)

100%|██████████| 166M/166M [00:05<00:00, 34.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mariaherrerot/idrid-dataset/versions/1


In [5]:
! rm -rf retinal-disease-classification
!git clone -b develop https://github.com/Tejashree-Khot/medsiglip-448-finetune.git
%cd /content/medsiglip-448-finetune/src

Cloning into 'medsiglip-448-finetune'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/medsiglip-448-finetune/src'
/content


In [6]:
! ls /kaggle/input/idrid-dataset

ls: cannot access '/kaggle/input/idrid-dataset': No such file or directory


In [ ]:
! ls /kaggle/input/idrid-dataset/Imagenes/Imagenes/IDRiD_001.jpg

In [ ]:
model = AutoModel.from_pretrained("google/medsiglip-448")
processor = AutoProcessor.from_pretrained("google/medsiglip-448")
model.to(DEVICE)

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

In [ ]:
imgs = [
    Image.open("/kaggle/input/idrid-dataset/Imagenes/Imagenes/IDRiD_001.jpg").convert("RGB"),
    Image.open("/kaggle/input/idrid-dataset/Imagenes/Imagenes/IDRiD_002.jpg").convert("RGB")
    ]

In [ ]:
def resize(image):
    return Image.fromarray(
        tf_resize(
            images=image, size=[448, 448], method='bilinear', antialias=False
        ).numpy().astype(np.uint8)
    )


resized_imgs = [resize(img) for img in imgs]

texts = [
    "retina is having Clinically_Significant_Macular_Edema",
    "retina is having No_DR",
    "retina is normal",
    "retina is having Mild_Moderate_NPDR",
    "retina is having Severe_PDR"
]

In [ ]:
inputs = processor(text=texts, images=resized_imgs, padding="max_length", return_tensors="pt").to(DEVICE)

In [ ]:
inputs["pixel_values"].shape, inputs["input_ids"].shape

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = torch.softmax(logits_per_image, dim=1)

for n_img, img in enumerate(imgs):
    print(f"{img.size}\n")
    # display(img)  # Note this is an IPython function that will only work in a Jupyter notebook environment
    for i, label in enumerate(texts):
        print(f"{probs[n_img][i]:.2%} that image is '{label}'")

# Get the image and text embeddings
print(f"image embeddings: {outputs.image_embeds}")
print(f"text embeddings: {outputs.text_embeds}")

### Plot embeddings

In [ ]:
# Combine image and text embeddings
embeddings = torch.cat((outputs.image_embeds, outputs.text_embeds), dim=0).cpu().numpy()

# Create labels for the embeddings
labels = ["Image 1", "Image 2"] + texts

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=2)
embeddings_tsne = tsne.fit_transform(embeddings)

# Plot the t-SNE results
plt.figure(figsize=(10, 8))

# Separate image and text embeddings for plotting
image_embeddings_tsne = embeddings_tsne[:len(imgs)]
text_embeddings_tsne = embeddings_tsne[len(imgs):]

# Plot image embeddings
plt.scatter(image_embeddings_tsne[:, 0], image_embeddings_tsne[:, 1], label='Images', marker='o')

# Plot text embeddings
plt.scatter(text_embeddings_tsne[:, 0], text_embeddings_tsne[:, 1], label='Texts', marker='x')


# Annotate points with labels
for i, label in enumerate(labels):
    plt.annotate(label, (embeddings_tsne[i, 0], embeddings_tsne[i, 1]))

plt.title('t-SNE of Image and Text Embeddings')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.grid(True)
plt.legend() # Add legend
plt.show()